In [1]:
!pip install google-search-results gradio

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=feacca00721cc72e6b6c83185537ff4a0761f05e240f1ee3d5714a0eb8a8cd2d
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [11]:
from serpapi import GoogleSearch
import csv
import gradio as gr

# 🔴 PUT YOUR API KEY HERE
API_KEY = "0ffe04e26e6b1ed0e8848507335cb7163f838f1280cd849e2ca1e728fde5eb71"


# ----------------------------
# API CRAWLER (SerpAPI)
# ----------------------------
def crawl_products(query):
    params = {
        "engine": "google_shopping",
        "q": query,
        "api_key": API_KEY,
        "hl": "en",
        "gl": "in"
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    products = []

    query_words = query.lower().split()

    for item in results.get("shopping_results", []):
        title = item.get("title", "").lower()

        # ✅ CONDITION 1: must contain all query words
        if not all(word in title for word in query_words):
            continue

        # ✅ CONDITION 2: avoid unwanted variants
        unwanted = ["pro", "max", "plus", "ultra"]
        if any(word in title for word in unwanted if word not in query_words):
            continue

        products.append({
            "title": item.get("title", "N/A"),
            "price": item.get("price", "N/A"),
            "rating": item.get("rating", "N/A"),
            "link": item.get("link", ""),
            "source": item.get("source", "Google Shopping")
        })

        if len(products) >= 10:
            break

    return products




# ----------------------------
# JSON → CSV
# ----------------------------
def json_to_csv(data, filename="products.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow(["Title", "Price", "Rating", "Link", "Source"])

        for item in data:
            writer.writerow([
                item["title"],
                item["price"],
                item["rating"],
                item["link"],
                item["source"]
            ])

    return filename


# ----------------------------
# MAIN FUNCTION
# ----------------------------
def search_products(query):
    data = crawl_products(query)

    if not data:
        return [], None

    filename = json_to_csv(data)

    return data, filename


# ----------------------------
# UI FUNCTION
# ----------------------------
def ui_function(query):
    data, file = search_products(query)

    if not data:
        return [["No data found", "-", "-", "-", "-"]], None

    table_data = [
        [d["title"], d["price"], d["rating"], d["link"], d["source"]]
        for d in data
    ]

    return table_data, file


# ----------------------------
# GRADIO UI
# ----------------------------
interface = gr.Interface(
    fn=ui_function,
    inputs=gr.Textbox(label="Enter Product Name"),
    outputs=[
        gr.Dataframe(headers=["Title", "Price", "Rating", "Link", "Source"]),
        gr.File(label="Download CSV")
    ],
    title="🛒 E-commerce Price Tracker (API-Based)",
    description="Search products using SerpAPI (No blocking 🚀)"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7851367abfd646367c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
